# Minería de Datos — Sesión 4
## Análisis exploratorio I: perfilar un archivo que no se conoce
### Universidad Distrital · 26160 · grupo 020‑81 · jueves 20 de agosto de 2026

El sábado pasado el problema era **conseguir** los datos y unirlos sin romperlos. Hoy el archivo ya
está armado y la pregunta es otra: **¿qué hay aquí adentro?**

Esta es la fase 2 de CRISP‑DM en serio. No es correr `describe()` y seguir: es salir de la sesión
con una lista escrita de **lo que hay que arreglar** y **lo que ya se puede preguntar**.

> **Regla de la sesión:** ningún gráfico todavía. Los gráficos son el sábado. Hoy se perfila con
> números, porque un gráfico bonito sobre una columna mal entendida es la forma más rápida de
> equivocarse con confianza.

---
## 1. El archivo de hoy

Registro Académico entrega **siete años de matrículas**, 2019 a 2025. La celda simula ese archivo
para que el notebook corra solo; en su proyecto es `pd.read_csv(...)` y nada más.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(20260820)
n = 6000

periodos = [f"{a}-{s}" for a in range(2019, 2026) for s in (1, 2)]
periodo  = rng.choice(periodos, n)
anio     = np.array([int(p[:4]) for p in periodo])

programa = rng.choice(["Sistemas", "Industrial", "Catastral", "Electrónica"], n, p=[.34, .26, .22, .18])
jornada  = rng.choice(["Diurna", "Nocturna"], n, p=[.55, .45])
nocturna = (jornada == "Nocturna")

municipio = rng.choice(
    ["Bogotá", "Soacha", "Chía", "Zipaquirá", "Mosquera", "Facatativá", "Madrid", "Cajicá", "Funza", "Sibaté"],
    n, p=[.62, .09, .05, .04, .04, .04, .03, .03, .03, .03])

edad = np.where(nocturna, rng.integers(20, 40, n), rng.integers(17, 26, n)).astype(float)
edad[rng.random(n) < 0.03] = 99          # 99 = "no informa", y NO es una edad

estrato = rng.choice([1, 2, 3, 4, 5], n, p=[.15, .38, .30, .12, .05]).astype(float)
estrato[rng.random(n) < 0.04] = 9        # 9 = "no informa", y NO es un estrato

promedio = np.where(nocturna, rng.normal(3.30, 0.45, n), rng.normal(3.75, 0.40, n))
promedio = np.clip(promedio, 1.5, 5.0).round(2)

creditos = rng.integers(9, 22, n)
creditos = np.where(anio == 2020, np.maximum(creditos - 5, 6), creditos)   # el semestre de la pandemia

horas = np.where(nocturna, rng.integers(20, 49, n), rng.integers(0, 10, n))

ingreso = np.round(rng.lognormal(np.log(2_400_000), 0.55, n), -3)

distancia = np.clip(rng.gamma(2.0, 5.0, n), 0.5, 60).round(1)

apoyo = rng.choice(["Sí", "No"], n, p=[.28, .72])

z = (-1.90
     - 1.50 * (promedio - 3.5)
     + 0.020 * horas
     + 0.015 * distancia
     - 0.35 * (apoyo == "Sí"))
retiro = rng.random(n) < 1 / (1 + np.exp(-z))

base = pd.DataFrame({
    "id":                   np.arange(100001, 100001 + n),
    "periodo":              periodo,
    "programa":             programa,
    "jornada":              jornada,
    "municipio":            municipio,
    "edad":                 edad,
    "estrato":              estrato,
    "promedio":             promedio,
    "creditos_inscritos":   creditos,
    "horas_trabajo_semana": horas,
    "ingreso_familiar":     ingreso,
    "distancia_km":         distancia,
    "apoyo_financiero":     apoyo,
    "retiro":               np.where(retiro, "Sí", "No"),
})

base.loc[rng.random(n) < 0.12, "ingreso_familiar"] = np.nan   # faltante de verdad
base.loc[rng.random(n) < 0.05, "distancia_km"]     = np.nan

df = (pd.concat([base, base.sample(30, random_state=11)], ignore_index=True)
        .sample(frac=1, random_state=11)
        .reset_index(drop=True))

df.shape

---
## 2. Las cuatro preguntas, otra vez — y qué se hace con la respuesta

Ya las conocen. Lo nuevo es la disciplina: **cada respuesta se anota**, y de esa lista sale el
trabajo de las semanas 4 y 5.

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

### Lea el `describe()` buscando pelea, no confirmación

Tres cosas saltan si se mira con desconfianza:

| Qué se ve | Qué significa |
|---|---|
| `edad` llega a **99** | Nadie de 99 años está matriculado. Es un **código de «no informa»** escrito como número. |
| `estrato` llega a **9** | Igual: no existe el estrato 9. Es una categoría disfrazada de número. |
| `ingreso_familiar`: la **media es mucho mayor que la mediana** | La distribución está sesgada. El promedio de esa columna no describe a nadie. |

**Ninguna de las tres la detecta `isna()`.** Por eso la fase 2 no se puede automatizar del todo.

---
## 3. Cuántos valores distintos tiene cada columna

`nunique()` es la pregunta más barata y la que más rápido clasifica una tabla: identificadores,
categóricas, numéricas de verdad.

In [ ]:
df.nunique().sort_values()

- Una columna con **tantos valores distintos como filas** es un identificador: `id`. No se modela
  con ella, y meterla en un modelo es una fuga silenciosa.
- Entre 2 y ~20 valores: **categórica**, aunque esté escrita con números (`estrato`).
- Cientos o miles de valores numéricos: **continua** (`ingreso_familiar`, `distancia_km`).

> **La trampa de siempre:** `estrato` es `float64` y `describe()` le calculó una media de 2,7.
> Un estrato promedio de 2,7 **no significa nada**. El tipo que pandas asigna no es el tipo que la
> variable tiene.

---
## 4. Las categóricas, una por una

In [ ]:
for col in ["programa", "jornada", "apoyo_financiero", "retiro"]:
    print(f"--- {col}")
    print(df[col].value_counts(normalize=True).round(3))
    print()

La última es **la tasa base**: se retira alrededor de una quinta parte. Ese número se anota en
grande —es contra lo que se comparará cualquier modelo en septiembre—.

Ahora una categórica con más niveles:

In [ ]:
df["municipio"].value_counts()

Bogotá se lleva casi dos terceras partes y hay una cola de municipios con pocos casos. Eso tiene una
consecuencia concreta para la semana 5: **las categorías con muy pocos casos hay que agruparlas**
en «Otros», o el modelo aprende de ruido. No se decide hoy, pero se anota.

---
## 5. Los faltantes: cuántos, y sobre todo **de quién**

Que falte un 12 % no dice nada. Lo que importa es si falta **al azar** o falta en un grupo concreto.

In [ ]:
print("faltantes por columna:")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("porcentaje:", (df["ingreso_familiar"].isna().mean() * 100).round(1), "%")

In [ ]:
# ¿el ingreso falta más en unos grupos que en otros?
falta = df["ingreso_familiar"].isna()

print("estudiantes por jornada:")
print(df.groupby("jornada").size())
print()
print("proporción de ingreso faltante, por jornada:")
print(falta.groupby(df["jornada"]).mean().round(3))
print()
print(falta.groupby(df["estrato"]).mean().round(3))

> **Por qué esto importa tanto.** Si el ingreso faltara sobre todo en los estratos bajos, rellenar
> con la mediana **inventaría plata** justo en el grupo que se quiere estudiar, y el modelo saldría
> optimista con la gente que peor está. Aquí falta parejo, así que rellenar es defendible. Esa
> decisión —y su justificación— es la semana 5.

---
## 6. Duplicados

In [ ]:
print("filas duplicadas completas :", df.duplicated().sum())
print("ids repetidos              :", df["id"].duplicated().sum())

Las dos preguntas son distintas y hay que hacer las dos. Un `id` repetido con datos distintos **no
es un duplicado**: es un estudiante con dos registros, y decidir cuál vale es una decisión de
negocio, no de programación.

---
## 7. Valores atípicos: los que son error y los que son el hallazgo

Dos reglas de dedo. Ninguna decide nada por sí sola.

In [ ]:
def atipicos_iqr(s):
    # cuántos valores caen fuera de 1,5 rangos intercuartílicos
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    bajo, alto = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < bajo) | (s > alto)).sum(), round(bajo, 2), round(alto, 2)


for col in ["edad", "promedio", "ingreso_familiar", "distancia_km", "horas_trabajo_semana"]:
    n_at, bajo, alto = atipicos_iqr(df[col])
    print(f"{col:22s} atípicos={n_at:5d}   rango normal=[{bajo}, {alto}]")

Y ahora la parte que no es mecánica — **mirar quiénes son**:

In [ ]:
print(df.loc[df["edad"] > 80, "edad"].value_counts())      # ¿son personas o es un código?
print()
print(df.loc[df["ingreso_familiar"] > 15_000_000,
             ["programa", "estrato", "ingreso_familiar"]].head(8))

| Columna | El atípico es… | Qué se hace |
|---|---|---|
| `edad = 99` | **un error de codificación** | se vuelve `NaN`: no es una edad |
| `estrato = 9` | **un error de codificación** | se vuelve `NaN` |
| `ingreso_familiar` muy alto | **un dato real y poco frecuente** | se conserva; quizá se transforme |

> **La frase de la sesión:** un atípico no se borra porque sea grande. Se borra porque **es falso**.
> Borrar los ingresos altos sería borrar exactamente la parte de la población que hace que la
> pregunta tenga sentido.

---
## 8. Corregir lo que ya sabemos que está mal

Dos líneas. El resto de la limpieza es la semana 5.

In [ ]:
d = df.copy()
d.loc[d["edad"] == 99, "edad"] = np.nan
d.loc[d["estrato"] == 9, "estrato"] = np.nan

print("edad    — máximo real:", d["edad"].max(), "| faltantes:", int(d["edad"].isna().sum()))
print("estrato — valores    :", sorted(d["estrato"].dropna().unique()))
print()
print(d[["edad", "estrato"]].describe().round(2))

Compare esa media de edad con la del `describe()` de la sección 2. **Los datos no cambiaron:**
cambió lo que sabemos de ellos. Ese es el oficio.

---
## 9. La función de perfilado — esto se lo llevan al proyecto

Todo lo anterior, en una sola tabla. Es lo primero que se le corre a cualquier archivo nuevo.

In [ ]:
def perfilar(df):
    filas = []
    for col in df.columns:
        s = df[col]
        fila = {
            "columna":   col,
            "tipo":      str(s.dtype),
            "distintos": s.nunique(),
            "faltan_%":  round(s.isna().mean() * 100, 1),
            "moda":      s.mode().iloc[0] if not s.mode().empty else None,
        }
        if pd.api.types.is_numeric_dtype(s):
            fila["min"]     = round(s.min(), 2)
            fila["mediana"] = round(s.median(), 2)
            fila["media"]   = round(s.mean(), 2)
            fila["max"]     = round(s.max(), 2)
        filas.append(fila)
    return pd.DataFrame(filas).set_index("columna")


perfilar(d)

**Cómo se lee esta tabla, en tres pasadas:**

1. **Columna `faltan_%`** — de mayor a menor. Todo lo que pase del 40 % es candidato a descartarse.
2. **`distintos`** — el 1 es una columna constante (sirve para nada); el que iguala el número de
   filas es un identificador.
3. **`media` contra `mediana`** — si se separan mucho, la columna está sesgada y el promedio miente.

---
## 10. Lo que se lleva de hoy

1. `describe()` se lee **buscando lo imposible**: máximos, mínimos y medias que no pueden ser.
2. `isna()` solo ve los faltantes que pandas reconoció. **99 y 9 eran faltantes** y no aparecían.
3. Un faltante importa por **quién** lo tiene, no por cuántos son.
4. Un atípico se elimina cuando es **falso**, no cuando es grande.
5. `nunique()` separa identificadores, categóricas y continuas en un segundo.
6. El perfilado termina en **una lista escrita** de qué arreglar. Sin esa lista, la fase 2 no ocurrió.

---
## 11. Trabajo autónomo — entra en el taller de la semana

Sobre **cada uno de sus cuatro candidatos**:

1. Corra `perfilar()` y pegue la salida.
2. Escriba **tres problemas** que encontró y qué haría con cada uno.
3. Diga cuál es la **columna objetivo** y su reparto —`value_counts(normalize=True)`—.
4. Una frase: ¿este archivo aguanta el semestre, sí o no, y por qué?

El taller se resuelve el sábado en clase y cierra el **domingo 30 de agosto, 11:59 p. m.**